In [1]:
!apt-get -qq -y install espeak-ng > /dev/null 2>&1
!pip install -q soundfile huggingface_hub loguru regex numpy addict num2words phonemizer espeakng-loader
!pip install -q --no-cache-dir git+https://github.com/Bindkushal/indic-g2p.git
!pip install -q --no-cache-dir git+https://github.com/Bindkushal/indic-voice.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.8/103.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.4/213.4 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 31.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 23.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metada

In [2]:
import subprocess
subprocess.run(['apt-get', 'install', '-y', 'espeak-ng', 'espeak-ng-data'], check=True)

CompletedProcess(args=['apt-get', 'install', '-y', 'espeak-ng', 'espeak-ng-data'], returncode=0)

In [3]:
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('Token set:', os.environ['HF_TOKEN'][:10], '...')

Token set: hf_WoORWQW ...


In [4]:
import espeakng_loader
import os

# Correct API based on dir() output
espeakng_loader.make_library_available()  # This likely sets everything up

data_path = espeakng_loader.get_data_path()
lib_path = espeakng_loader.get_library_path()

print("Data path:", data_path)
print("Lib path:", lib_path)

os.environ['ESPEAK_DATA_PATH'] = str(data_path)

Data path: /usr/local/lib/python3.12/dist-packages/espeakng_loader/espeak-ng-data
Lib path: /usr/local/lib/python3.12/dist-packages/espeakng_loader/libespeak-ng.so


In [5]:
import espeakng_loader
print(dir(espeakng_loader))

['Path', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'ctypes', 'get_data_path', 'get_library_path', 'load_library', 'make_library_available', 'os', 'platform']


In [7]:
from indicvoice import IndicPipeline
import soundfile as sf
from IPython.display import display, Audio

# Load pipeline
pipeline = IndicPipeline(
    lang_code='hi',
    repo_id='Bindkushal/IndicVoice-82M'
)
print('Pipeline loaded!')

# Generate audio
story = '''
एक छोटी चिड़िया थी। वह रोज़ सुबह गाना गाती थी।
एक दिन बारिश आई। चिड़िया ने पेड़ के नीचे छुपकर इंतज़ार किया।
बारिश रुकी तो इंद्रधनुष निकला। चिड़िया खुशी से उड़ गई।
'''

for i, (gs, ps, audio) in enumerate(pipeline(story, voice='hf_alpha', speed=1)):
    print(f'Chunk {i}:', gs)
    print('Phonemes:', ps)
    display(Audio(data=audio, rate=24000, autoplay=i==0))
    sf.write(f'hindi_story_{i}.wav', audio, 24000)

Pipeline loaded!
Chunk 0: एक छोटी चिड़िया थी। वह रोज़ सुबह गाना गाती थी।
Phonemes: ˈeːk cʰˈoːʈi cˈɪr.ɪjˌaː tʰi ʋˌəh ɾˈoːz sˈʊbəh ɡˈaːnaː ɡˈaːti tʰi


Chunk 1: एक दिन बारिश आई। चिड़िया ने पेड़ के नीचे छुपकर इंतज़ार किया।
Phonemes: ˈeːk dˈɪn bˈaːɾɪʃ ˈaːi cˈɪr.ɪjˌaː nˈeː pˈeːr. keː nˈiːceː cʰˈʊpəkˌəɾ ˌĩtzˈaːɾ kˈɪjaː


Chunk 2: बारिश रुकी तो इंद्रधनुष निकला। चिड़िया खुशी से उड़ गई।
Phonemes: bˈaːɾɪʃ ɾˈʊki tˈoː ˈĩdɾədʰnʊʂ nˈɪklˌaː cˈɪr.ɪjˌaː kʰˈʊʃi seː ˈʊr. ɡˈʌi
